**Group members:** Wario Louis Maina-190124, Ongoma Derrick Omondi-182842, Muhoro Benedict Kuria-171534

Uniform Cost Search always expands the frontier node with the lowest cumulative path cost g(n). When the goal is removed from the frontier, the path found is optimal because all edge costs are positive.

## Step 1: Problem formulation (directed graph, distances in kilometres)

In [1]:
import heapq
import itertools

# Each two-way road is stored as two directed edges
ROADS = [
    ("Nairobi", "Limuru", 30), ("Nairobi", "Thika", 45), ("Nairobi", "Narok", 140),
    ("Limuru", "Naivasha", 60), ("Thika", "Murang'a", 40), ("Murang'a", "Nyeri", 55),
    ("Nyeri", "Nyahururu", 70), ("Naivasha", "Gilgil", 30), ("Gilgil", "Nakuru", 35),
    ("Nyahururu", "Nakuru", 85), ("Nakuru", "Londiani", 75), ("Londiani", "Kericho", 35),
    ("Londiani", "Eldoret", 85), ("Narok", "Bomet", 95), ("Bomet", "Kericho", 50),
    ("Kericho", "Kisumu", 90),
]

REGIONS = {
    "Nairobi": "Nairobi",
    "Limuru": "Central Highlands", "Thika": "Central Highlands",
    "Murang'a": "Central Highlands", "Nyeri": "Central Highlands",
    "Nyahururu": "Central Highlands",
    "Naivasha": "Rift Valley", "Gilgil": "Rift Valley", "Nakuru": "Rift Valley",
    "Londiani": "Rift Valley", "Kericho": "Rift Valley", "Eldoret": "Rift Valley",
    "Narok": "Rift Valley", "Bomet": "Rift Valley",
    "Kisumu": "Lake Victoria Basin",
}

INITIAL_STATE = "Nairobi"
GOAL_STATE = "Kisumu"


def build_graph(roads):
    graph = {}
    for a, b, km in roads:
        graph.setdefault(a, []).append((b, km))
        graph.setdefault(b, []).append((a, km))
    return graph


def goal_test(state):
    return state == GOAL_STATE


graph = build_graph(ROADS)
for town, edges in graph.items():
    print(f"{town:<10} -> {edges}")

Nairobi    -> [('Limuru', 30), ('Thika', 45), ('Narok', 140)]
Limuru     -> [('Nairobi', 30), ('Naivasha', 60)]
Thika      -> [('Nairobi', 45), ("Murang'a", 40)]
Narok      -> [('Nairobi', 140), ('Bomet', 95)]
Naivasha   -> [('Limuru', 60), ('Gilgil', 30)]
Murang'a   -> [('Thika', 40), ('Nyeri', 55)]
Nyeri      -> [("Murang'a", 55), ('Nyahururu', 70)]
Nyahururu  -> [('Nyeri', 70), ('Nakuru', 85)]
Gilgil     -> [('Naivasha', 30), ('Nakuru', 35)]
Nakuru     -> [('Gilgil', 35), ('Nyahururu', 85), ('Londiani', 75)]
Londiani   -> [('Nakuru', 75), ('Kericho', 35), ('Eldoret', 85)]
Kericho    -> [('Londiani', 35), ('Bomet', 50), ('Kisumu', 90)]
Eldoret    -> [('Londiani', 85)]
Bomet      -> [('Narok', 95), ('Kericho', 50)]
Kisumu     -> [('Kericho', 90)]


## Step 2: Uniform Cost Search with step-by-step trace

In [2]:
def format_frontier(frontier):
    return "[" + ", ".join(f"{t}({c})" for c, _, t, _ in sorted(frontier)) + "]"


def uniform_cost_search(graph, start):
    counter = itertools.count()                       # tie-breaker: earlier-generated node first
    frontier = [(0, next(counter), start, [start])]   # (g, order, town, path)
    best_cost = {start: 0}                            # cheapest known cost to each town
    explored = set()
    step = 0

    print("=" * 78)
    print(f"UNIFORM COST SEARCH: {start} -> {GOAL_STATE}")
    print("=" * 78)

    while frontier:
        step += 1
        print(f"\nStep {step}")
        print(f"  Frontier before expansion: {format_frontier(frontier)}")
        cost, _, town, path = heapq.heappop(frontier)

        if town in explored:
            print(f"  Skipped {town} (g={cost}): already expanded with a lower cost")
            continue

        print(f"  Expand: {town} (cumulative cost g = {cost} km)")

        # Goal test is applied when the node is removed from the frontier
        if goal_test(town):
            print(f"  Goal test passed: {town} is the destination")
            return path, cost, step

        explored.add(town)

        for neighbour, km in graph[town]:
            new_cost = cost + km
            if neighbour in explored:
                print(f"    {neighbour}: g = {new_cost} pruned "
                      f"(already expanded at g = {best_cost[neighbour]})")
            elif neighbour not in best_cost or new_cost < best_cost[neighbour]:
                best_cost[neighbour] = new_cost
                heapq.heappush(frontier, (new_cost, next(counter), neighbour, path + [neighbour]))
                print(f"    {neighbour}: g = {new_cost} added to the frontier")
            else:
                print(f"    {neighbour}: g = {new_cost} pruned "
                      f"(cheaper path with g = {best_cost[neighbour]} already on the frontier)")

        print(f"  Explored set: {sorted(explored)}")

    return None, float("inf"), step


path, total_cost, steps = uniform_cost_search(graph, INITIAL_STATE)

UNIFORM COST SEARCH: Nairobi -> Kisumu

Step 1
  Frontier before expansion: [Nairobi(0)]
  Expand: Nairobi (cumulative cost g = 0 km)
    Limuru: g = 30 added to the frontier
    Thika: g = 45 added to the frontier
    Narok: g = 140 added to the frontier
  Explored set: ['Nairobi']

Step 2
  Frontier before expansion: [Limuru(30), Thika(45), Narok(140)]
  Expand: Limuru (cumulative cost g = 30 km)
    Nairobi: g = 60 pruned (already expanded at g = 0)
    Naivasha: g = 90 added to the frontier
  Explored set: ['Limuru', 'Nairobi']

Step 3
  Frontier before expansion: [Thika(45), Naivasha(90), Narok(140)]
  Expand: Thika (cumulative cost g = 45 km)
    Nairobi: g = 90 pruned (already expanded at g = 0)
    Murang'a: g = 85 added to the frontier
  Explored set: ['Limuru', 'Nairobi', 'Thika']

Step 4
  Frontier before expansion: [Murang'a(85), Naivasha(90), Narok(140)]
  Expand: Murang'a (cumulative cost g = 85 km)
    Thika: g = 125 pruned (already expanded at g = 45)
    Nyeri: g = 140

## Step 3: Optimal path and its cost

In [3]:
def leg_distance(graph, a, b):
    return next(km for n, km in graph[a] if n == b)


print("=" * 65)
print("RESULT")
print("=" * 65)

if path is None:
    print("No route found.")
else:
    print(f"Optimal path : {' -> '.join(path)}")
    print(f"Nodes expanded (steps) : {steps}\n")
    print(f"{'Leg':<28}{'Region of destination':<24}{'Distance (km)':>13}")
    print("-" * 65)
    for a, b in zip(path, path[1:]):
        print(f"{a + ' -> ' + b:<28}{REGIONS[b]:<24}{leg_distance(graph, a, b):>13}")
    print("-" * 65)
    print(f"{'Total path cost':<52}{total_cost:>13}")

RESULT
Optimal path : Nairobi -> Limuru -> Naivasha -> Gilgil -> Nakuru -> Londiani -> Kericho -> Kisumu
Nodes expanded (steps) : 15

Leg                         Region of destination   Distance (km)
-----------------------------------------------------------------
Nairobi -> Limuru           Central Highlands                  30
Limuru -> Naivasha          Rift Valley                        60
Naivasha -> Gilgil          Rift Valley                        30
Gilgil -> Nakuru            Rift Valley                        35
Nakuru -> Londiani          Rift Valley                        75
Londiani -> Kericho         Rift Valley                        35
Kericho -> Kisumu           Lake Victoria Basin                90
-----------------------------------------------------------------
Total path cost                                               355
